In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from lightgbm import LGBMRegressor
import lightgbm as lgb

### Preprocessing

In [6]:
# load files
sales = pd.read_csv("../data/sales_train_validation.csv")
calendar = pd.read_csv("../data/calendar.csv")
sell_prices = pd.read_csv("../data/sell_prices.csv")

# convert date column to datetime
calendar["date"] = pd.to_datetime(calendar["date"]) 

# keep id columns separately
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales.columns if c.startswith("d_")]

# wide to long:
sales_long = sales.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name="d",
    value_name="sales"
)

# join calendar info
df = sales_long.merge(calendar, on="d", how="left")

# join prices using Walmart week key
df = df.merge(
    sell_prices,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

# date handling
df["date"] = pd.to_datetime(df["date"])

# reduce memory a bit
for col in df.select_dtypes(include=["int64"]).columns:
    df[col] = pd.to_numeric(df[col], downcast="integer")

for col in df.select_dtypes(include=["float64"]).columns:
    df[col] = pd.to_numeric(df[col], downcast="float")

# combine snap into single feature instead of state wise columns
df["snap"] = 0

df.loc[df["state_id"] == "CA", "snap"] = df["snap_CA"]
df.loc[df["state_id"] == "TX", "snap"] = df["snap_TX"]
df.loc[df["state_id"] == "WI", "snap"] = df["snap_WI"]

df = df.drop(columns=["snap_CA", "snap_TX", "snap_WI"]) # drop original snap columns

# keep event type only, event name is too sparse and not likely to be useful
df = df.drop(columns=["event_name_1", "event_name_2"])

# use just event type 1, event type 2 is too sparse. Fill NA with None to create new category for no event.
df["event_type"] = df["event_type_1"].fillna("None") 
df = df.drop(columns=["event_type_1", "event_type_2"]) # drop original event type columns

# keep categorical columns as category dtype for memory efficiency
cat_cols = [
    "item_id", "dept_id", "cat_id", "store_id", "state_id",
    # "event_name_1", "event_type_1", "event_name_2", "event_type_2",
    "weekday", "event_type"
]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")



df = df.drop(columns=["d"]) # we have date now, no need for d

# memory issues still persist, so we 
# keep last 500 days only
cutoff_date = df["date"].max() - pd.Timedelta(days=500)
df = df[df["date"] > cutoff_date]

# sort before feature engineering
df = df.sort_values(["store_id", "item_id", "date"]).reset_index(drop=True)


In [7]:
df.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'sales',
       'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'sell_price',
       'snap', 'event_type'],
      dtype='object')

In [8]:
df.head(5)

,id,item_id,dept_id,cat_id,store_id,state_id,sales,date,wm_yr_wk,weekday,wday,month,year,sell_price,snap,event_type
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,1,2014-12-12,11445,Friday,7,12,2014,2.24,0,None
1,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,2,2014-12-13,11446,Saturday,1,12,2014,2.24,0,None
2,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,2,2014-12-14,11446,Sunday,2,12,2014,2.24,0,None
3,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,1,2014-12-15,11446,Monday,3,12,2014,2.24,0,None
4,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,1,2014-12-16,11446,Tuesday,4,12,2014,2.24,0,None


### Feature Engineering

lag_7, lag_14, lag_28, lag_56: past sales values at fixed lags capturing weekly/monthly demand patterns.

rmean_7, rmean_14, rmean_28: rolling average of past sales capturing smoothed demand trends.

rstd_7, rstd_14, rstd_28: rolling standard deviation capturing demand volatility.

price_change: ratio of current price to previous price capturing price shocks.

price_mean_7: rolling average of past prices capturing pricing trends.

dayofweek: numeric day index capturing weekly seasonality.

month: month index capturing yearly seasonality.

weekofyear: week index capturing finer seasonal cycles.

is_weekend: binary flag capturing weekend demand effects.

store_sales_mean_7: rolling mean of total store demand capturing store-level trends.

cat_sales_mean_7: rolling mean of category demand capturing product-group trends.

In [9]:
# ensure sorted
df = df.sort_values(["store_id", "item_id", "date"])

# -------- LAG FEATURES --------
lags = [1, 7, 28]

for lag in lags:
    df[f"lag_{lag}"] = df.groupby(["store_id", "item_id"])["sales"].shift(lag)

# -------- ROLLING FEATURES --------
# not usnig any rolling std for now, likely noisy and not very helpful

g = df.groupby(["store_id", "item_id"])["sales"]

for window in [3, 7, 28]:
    df[f"rmean_{window}"] = (
        g.shift(1)
        .rolling(window)
        .mean()
    )




# -------- PRICE FEATURES --------
# price change (very important)
df["price_lag_1"] = df.groupby(["store_id", "item_id"])["sell_price"].shift(1)
df["price_change"] = df["sell_price"] / df["price_lag_1"]

# price momentum
df["price_mean_7"] = (
    df.groupby(["store_id", "item_id"])["sell_price"]
    .shift(1)
    .rolling(7)
    .mean()
)

# -------- CALENDAR FEATURES --------
df["dayofweek"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["weekofyear"] = df["date"].dt.isocalendar().week.astype("int")

# weekend flag
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)

# SNAP already exists → keep as is

# -------- HIERARCHICAL FEATURES (IMPORTANT BOOST) --------

# store-level demand
df["store_sales_mean_7"] = (
    df.groupby(["store_id", "date"])["sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

# category-level demand
df["cat_sales_mean_7"] = (
    df.groupby(["cat_id", "date"])["sales"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

# downcast new features to save memory
for col in df.select_dtypes(include=["float64"]).columns:
    df[col] = df[col].astype("float32")

# drop intermediate price lag column to save memory
df = df.drop(columns=["price_lag_1"])

# drop unnecessary columns (used to create better features, not needed anymore)
df = df.drop(columns=[ "wm_yr_wk", "weekday", "year"])

print(df.shape)

/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_73382/1697551663.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f"lag_{lag}"] = df.groupby(["store_id", "item_id"])["sales"].shift(lag)
/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_73382/1697551663.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f"lag_{lag}"] = df.groupby(["store_id", "item_id"])["sales"].shift(lag)
/var/folders/x9/fhf6q7xd0wjgg_92l4b1sykm0000gn/T/ipykernel_73382/1697551663.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass obser

(15245000, 26)


In [45]:
# using all data:
# 30,490 item-store series × 1,913 days ≈ 58,356,370 rows

# using last 500 days only:
# 30,490 item-store series × 500 days = 15,245,000 rows

In [10]:
df.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'sales',
       'date', 'wday', 'month', 'sell_price', 'snap', 'event_type', 'lag_1',
       'lag_7', 'lag_28', 'rmean_3', 'rmean_7', 'rmean_28', 'price_change',
       'price_mean_7', 'dayofweek', 'weekofyear', 'is_weekend',
       'store_sales_mean_7', 'cat_sales_mean_7'],
      dtype='object')

### Time Based Train Val Split

In [11]:
# cutoff 
cutoff = df["date"].max() - pd.Timedelta(days=28)

train_df = df[df["date"] <= cutoff]
val_df   = df[df["date"] > cutoff]

### Modelling

In [12]:
# drop target + identifiers you don’t want as features
drop_cols = ["sales", "date", "id"]

features = [c for c in df.columns if c not in drop_cols]
target = "sales"

In [13]:
# item level model, without tweedie objective, didnt perform well

# model = LGBMRegressor(
#     metric="rmse",
#     n_estimators=1000,
#     learning_rate=0.05,
#     num_leaves=255,
#     subsample=0.8,
#     colsample_bytree=0.8
# )

# cat_cols = [c for c in features if c in df.select_dtypes(include=["category"]).columns]
# model.fit(
#     train_df[features],
#     train_df[target],
#     eval_set=[(val_df[features], val_df[target])],
#     eval_metric="rmse",
#     categorical_feature=cat_cols,
#     callbacks=[
#         lgb.early_stopping(50),
#         lgb.log_evaluation(100)
#     ]
# )

In [14]:
# train a model for each store separately, use tweedie objective - better handling of sparse series 

models = {}

cat_cols_model = [c for c in features if c in df.select_dtypes(include=["category"]).columns]

for store in train_df["store_id"].unique():
    
    print(f"Training store: {store}")
    
    train_store = train_df[train_df["store_id"] == store]
    val_store   = val_df[val_df["store_id"] == store]
    
    model = LGBMRegressor(
        objective="tweedie",
        tweedie_variance_power=1.1,
        n_estimators=1000,
        learning_rate=0.05,
        num_leaves=255,
        subsample=0.8,
        colsample_bytree=0.8
    )
    
    model.fit(
        train_store[features],
        train_store[target],
        eval_set=[(val_store[features], val_store[target])],
        eval_metric="rmse",
        categorical_feature=cat_cols_model,
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(100)
        ]
    )
    
    models[store] = model

Training store: CA_1
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.035555 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4840
[LightGBM] [Info] Number of data points in the train set: 1439128, number of used features: 21
[LightGBM] [Info] Start training from score 0.362088
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 2.00481	valid_0's tweedie: 14.9034
Early stopping, best iteration is:
[83]	valid_0's rmse: 2.00392	valid_0's tweedie: 14.9031
Training store: CA_2
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightG

### Generate Predicitons

In [15]:
# retrain on full train data
models = {}

cat_cols_model = [c for c in features if c in df.select_dtypes(include=["category"]).columns]

for store in df["store_id"].unique():
    
    print(f"Retraining store on full data: {store}")
    
    train_store = df[df["store_id"] == store]
    
    model = LGBMRegressor(
        objective="tweedie",
        tweedie_variance_power=1.1,
        n_estimators=80, # average across stores in validation run 
        learning_rate=0.05,
        num_leaves=255,
        subsample=0.8,
        colsample_bytree=0.8
    )
    
    model.fit(
        train_store[features],
        train_store[target],
        categorical_feature=cat_cols_model
    )
    
    models[store] = model

Retraining store on full data: CA_1
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.040629 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4845
[LightGBM] [Info] Number of data points in the train set: 1524500, number of used features: 21
[LightGBM] [Info] Start training from score 0.364521
Retraining store on full data: CA_2
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, 

In [58]:
# RECURSIVE PREDICTION

# last date
last_date = df["date"].max()
future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=28)

# base ids
base = df[["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]].drop_duplicates()

# future frame
future_df = base.merge(pd.DataFrame({"date": future_dates}), how="cross")

# merge calendar + prices
future_df = future_df.merge(calendar, on="date", how="left")
future_df = future_df.merge(sell_prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")

# combine with history
full_df = pd.concat([df, future_df], ignore_index=True)
full_df = full_df.sort_values(["store_id", "item_id", "date"])

full_df["sales"] = full_df["sales"].astype("float64")

# -------- FIX: categorical alignment (only valid cols) --------
cat_cols_model = [c for c in features if c in df.select_dtypes(include=["category"]).columns]

for col in cat_cols_model:
    if col in full_df.columns:
        full_df[col] = full_df[col].astype("category")
        full_df[col] = full_df[col].cat.set_categories(df[col].cat.categories)

# -------- RECURSIVE LOOP --------
preds = []

for day in future_dates:
    
    g = full_df.groupby(["store_id", "item_id"])
    
    # --- lag features ---
    full_df["lag_1"] = g["sales"].shift(1)
    full_df["lag_7"] = g["sales"].shift(7)
    full_df["lag_28"] = g["sales"].shift(28)
    
    # --- rolling ---
    full_df["rmean_7"] = g["sales"].shift(1).rolling(7).mean()
    full_df["rmean_28"] = g["sales"].shift(1).rolling(28).mean()
    
    # --- calendar features ---
    full_df["dayofweek"] = full_df["date"].dt.dayofweek
    full_df["month"] = full_df["date"].dt.month
    full_df["is_weekend"] = (full_df["dayofweek"] >= 5).astype(int)
    
    # --- price features ---
    full_df["price_lag_1"] = g["sell_price"].shift(1)
    full_df["price_change"] = full_df["sell_price"] / full_df["price_lag_1"]
    full_df["price_change"] = full_df["price_change"].replace([np.inf, -np.inf], 1).fillna(1)
    
    # --- select current day ---
    mask = full_df["date"] == day
    X = full_df.loc[mask, features]
    
    # --- store-wise prediction ---
    y_pred = np.zeros(len(X))
    
    for store in X["store_id"].unique():
        mask_store = X["store_id"] == store
        y_pred[mask_store] = models[store].predict(X.loc[mask_store])
    
    # write back for recursion
    full_df.loc[mask, "sales"] = y_pred
    
    preds.append(y_pred)

# final predictions
preds = np.stack(preds, axis=1)

print(preds.shape)  # should be (30490, 28)

(30490, 28)


### Generating submission file for base line model eval

In [59]:
# load sample submission for structure
sample_sub = pd.read_csv("sample_submission.csv")

# FIX: remove existing suffix to get base ids
base_ids = sales["id"].str.replace("_validation", "", regex=False)

# create prediction df
preds_df = pd.DataFrame(preds, columns=[f"F{i}" for i in range(1, 29)])
preds_df["id"] = base_ids.values

# build validation + evaluation ids correctly
preds_val = preds_df.copy()
preds_val["id"] = preds_val["id"] + "_validation"

preds_eval = preds_df.copy()
preds_eval["id"] = preds_eval["id"] + "_evaluation"

# combine
submission = pd.concat([preds_val, preds_eval], axis=0)

# reorder to match sample submission
submission = submission.set_index("id").loc[sample_sub["id"]].reset_index()

# save
submission.to_csv("submission2.csv", index=False)

# sanity check
print(submission.shape)  # should be (60980, 29)

(60980, 29)


### Hyperparameter Tuning

In [ ]:
# ideally we'd like to use hyperparameter tuning with validation set and use wmrsse to select best model, but memory issues prevent that for now. We can do that later if we have time.

In [56]:
# ---------- LOAD ----------
sales_val  = pd.read_csv("sales_train_validation.csv")
sales_eval = pd.read_csv("sales_train_evaluation.csv")
weights_df = pd.read_csv("weights_validation.csv")

# ---------- BUILD PRED DF ----------
ids = sales_val["id"].values
pred_cols = [f"d_{i}" for i in range(1914, 1942)]

preds_df = pd.DataFrame(preds, columns=pred_cols)
preds_df.insert(0, "id", ids)

# ---------- TRUE VALUES ----------
true_df = sales_eval[["id"] + pred_cols]

# ---------- TRAIN HISTORY ----------
train_cols = [f"d_{i}" for i in range(1, 1914)]
train_df = sales_val[["id"] + train_cols]

# ---------- ENSURE ALIGNMENT (NO FILTERING) ----------
# ---------- ALIGN IDS (FINAL FIX) ----------

# convert validation ids → evaluation ids for true_df lookup
ids_eval = [i.replace("_validation", "_evaluation") for i in ids]

# align true values (evaluation set)
true_df = (
    true_df
    .set_index("id")
    .loc[ids_eval]
    .reset_index()
)

# convert back to validation format so all ids match downstream
true_df["id"] = true_df["id"].str.replace("_evaluation", "_validation")

# align predictions (already validation ids)
preds_df = (
    preds_df
    .set_index("id")
    .loc[ids]
    .reset_index()
)

# ---------- TO NUMPY ----------
train_np = train_df.iloc[:, 1:].values
true_np  = true_df.iloc[:, 1:].values
pred_np  = preds_df.iloc[:, 1:].values
ids_np   = ids

# ---------- RMSSE ----------
def rmsse(train, true, pred):
    denom = np.mean((train[1:] - train[:-1]) ** 2)
    num   = np.mean((true - pred) ** 2)
    return np.sqrt(num / (denom + 1e-8))   # fixed

# ---------- WEIGHTS (CORRECT ID FORMAT) ----------
w = weights_df[weights_df["Level_id"] == "Level12"].copy()

w["id"] = (
    w["Agg_Level_1"] + "_" +
    w["Agg_Level_2"] +
    "_validation"
)

weight_map = dict(zip(w["id"], w["Weight"]))

# ---------- WRMSSE ----------
score = 0.0

for i, id_ in enumerate(ids_np):
    w_i = weight_map.get(id_)
    if w_i is None:
        continue
    
    s = rmsse(train_np[i], true_np[i], pred_np[i])
    score += w_i * s

print("WRMSSE:", score)

WRMSSE: 1.62098385364378


In [ ]:
# use wrmsse for hyperparam tuning